# Notebook 5 . Cari dan Hapus Duplikat

Temukan baris ganda persis atau baris dengan ID/kombinasi kolom yang sama. Pilih cara menyimpan salah satu salinan. Periksa laporan sebelum memakai hasil.

In [ ]:
from pathlib import Path
import pandas as pd

# PENGATURAN YANG BISA DIUBAH
Input_CSV = Path("Input/input.csv")
Separator_CSV = ","  # Ganti menjadi ";" jika CSV memakai titik koma
Kolom_Kunci = ["id"]  # Gunakan [] untuk mencari duplikat baris yang seluruh kolomnya sama
Baris_Duplikat_Dipertahankan = "pertama"  # "pertama" atau "terakhir"
Output_CSV = Path("Output/hasil_tanpa_duplikat.csv")

def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Periksa lalu deduplikasi

In [ ]:
source = Input_CSV if Input_CSV.is_absolute() else BASE_DIR / Input_CSV
if not source.exists(): raise FileNotFoundError(f"File tidak ditemukan: {source}")
df = pd.read_csv(source, sep=Separator_CSV, dtype="string", encoding="utf-8-sig")
df.columns = df.columns.astype(str).str.strip()
subset = Kolom_Kunci or None
if subset:
    missing = [col for col in subset if col not in df]
    if missing: raise KeyError(f"Kolom kunci tidak ditemukan: {missing}")
    duplicates = df.duplicated(subset=subset, keep=False)
else:
    duplicates = df.duplicated(keep=False)
print(f"Baris yang termasuk kelompok duplikat: {int(duplicates.sum()):,}")
display(df.loc[duplicates].head(30))
pilihan_duplikat = {"pertama": "first", "terakhir": "last"}
if Baris_Duplikat_Dipertahankan not in pilihan_duplikat:
    raise ValueError("Baris_Duplikat_Dipertahankan harus 'pertama' atau 'terakhir'.")
result = df.drop_duplicates(subset=subset, keep=pilihan_duplikat[Baris_Duplikat_Dipertahankan])
print(f"Baris awal: {len(df):,}; hasil: {len(result):,}; dihapus: {len(df)-len(result):,}")


In [ ]:
output = Output_CSV if Output_CSV.is_absolute() else BASE_DIR / Output_CSV
output.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output, index=False, encoding="utf-8-sig", sep=Separator_CSV)
print(f"Tersimpan: {output.resolve()}")